# Experimental streamed moments + eager tied storage + activation offload

Distinct memory candidate, not executed on GPU. Native fixed task and true
CE/entropy/argmax/rewind outputs retained. Chunk256 head projections are replayed
in backward without full-token logits or dZ storage; two-pass original Triton
moments compute stable softmax summaries. Exact attested original half tiedtable
skips redundant FP32 preparation allocation and embedding outputs widen to FP32.
Native backbone saves use same-dtype pinned host buffers for eligible activations,
without enabling any protected built-in offload/checkpointing/task flag.

No GPU compilation, quality, time, peak-memory, original-share or admission claim.
Local evidence:31 NumPy moment cases;26 eager-prepare differential cases;47,988
small-layout offload safety cases; Python compile and source/payload hashes.
Numerical fixture/tolerances are retained; preflight lifecycle assertion adds
offload restoration and positive actual pack/unpack checks, and a CUDA-copy
component probe runs first. Finite LoRA numeric failures remain visible as
warnings; this is not strict jury admission. Run after artifact approval.


In [ ]:
import os, base64, hashlib, json, tempfile
from pathlib import Path
assert os.environ.get('SOUP_REPO') and os.environ.get('SOUP_RESULTS')
assert Path('/kaggle/working').is_dir()
MODE = 'streamed_moment_eager_offload_v1'
PAYLOAD = {'README.md': 'IyBFeHBlcmltZW50YWwgc3RyZWFtZWQgbW9tZW50cyArIGVhZ2VyIHRpZWQgc3RvcmFnZSArIGFjdGl2YXRpb24gb2ZmbG9hZAoKRGlzdGluY3QgbWVtb3J5IGNhbmRpZGF0ZSwgbm90IGV4ZWN1dGVkIG9uIEdQVS4gTmF0aXZlIGZpeGVkIHRhc2sgYW5kIHRydWUKQ0UvZW50cm9weS9hcmdtYXgvcmV3aW5kIG91dHB1dHMgcmV0YWluZWQuIENodW5rMjU2IGhlYWQgcHJvamVjdGlvbnMgYXJlIHJlcGxheWVkCmluIGJhY2t3YXJkIHdpdGhvdXQgZnVsbC10b2tlbiBsb2dpdHMgb3IgZFogc3RvcmFnZTsgdHdvLXBhc3Mgb3JpZ2luYWwgVHJpdG9uCm1vbWVudHMgY29tcHV0ZSBzdGFibGUgc29mdG1heCBzdW1tYXJpZXMuIEV4YWN0IGF0dGVzdGVkIG9yaWdpbmFsIGhhbGYgdGllZHRhYmxlCnNraXBzIHJlZHVuZGFudCBGUDMyIHByZXBhcmF0aW9uIGFsbG9jYXRpb24gYW5kIGVtYmVkZGluZyBvdXRwdXRzIHdpZGVuIHRvIEZQMzIuCk5hdGl2ZSBiYWNrYm9uZSBzYXZlcyB1c2Ugc2FtZS1kdHlwZSBwaW5uZWQgaG9zdCBidWZmZXJzIGZvciBlbGlnaWJsZSBhY3RpdmF0aW9ucywKd2l0aG91dCBlbmFibGluZyBhbnkgcHJvdGVjdGVkIGJ1aWx0LWluIG9mZmxvYWQvY2hlY2twb2ludGluZy90YXNrIGZsYWcuCgpObyBHUFUgY29tcGlsYXRpb24sIHF1YWxpdHksIHRpbWUsIHBlYWstbWVtb3J5LCBvcmlnaW5hbC1zaGFyZSBvciBhZG1pc3Npb24gY2xhaW0uCkxvY2FsIGV2aWRlbmNlOjMxIE51bVB5IG1vbWVudCBjYXNlczsyNiBlYWdlci1wcmVwYXJlIGRpZmZlcmVudGlhbCBjYXNlczs0Nyw5ODgKc21hbGwtbGF5b3V0IG9mZmxvYWQgc2FmZXR5IGNhc2VzOyBQeXRob24gY29tcGlsZSBhbmQgc291cmNlL3BheWxvYWQgaGFzaGVzLgpOdW1lcmljYWwgZml4dHVyZS90b2xlcmFuY2VzIGFyZSByZXRhaW5lZDsgcHJlZmxpZ2h0IGxpZmVjeWNsZSBhc3NlcnRpb24gYWRkcwpvZmZsb2FkIHJlc3RvcmF0aW9uIGFuZCBwb3NpdGl2ZSBhY3R1YWwgcGFjay91bnBhY2sgY2hlY2tzLCBhbmQgYSBDVURBLWNvcHkKY29tcG9uZW50IHByb2JlIHJ1bnMgZmlyc3QuIEZpbml0ZSBMb1JBIG51bWVyaWMgZmFpbHVyZXMgcmVtYWluIHZpc2libGUgYXMKd2FybmluZ3M7IHRoaXMgaXMgbm90IHN0cmljdCBqdXJ5IGFkbWlzc2lvbi4gUnVuIGFmdGVyIGFydGlmYWN0IGFwcHJvdmFsLgo=', 'runtime_api_reference.json': 'ewogICJwcm92ZW5hbmNlIjogIlBhcmVudCBhZ2VudCBhdXRoZW50aWNhdGVkIHJlYWQtb25seSBydW50aW1lIEdFVCwgdmVyaWZpZWQgVExTLCAyMDI2LTEwLTMwIGFyb3VuZCAyMDo1MCBVVEM7IG5vIHJlcXVlc3QgYnkgdGhpcyBhZGFwdGVyIGF1dGhvciBhbmQgbm8gR1BVIGpvYiIsCiAgImltYWdlIjogInNoYTI1NjpkMjgxMzlmNjljNTZhYzg5N2IyNjQ4N2YwZjdkODIzNmQ3MDFmMDMwMjE0MzQxYjUxMTNiNDcxNDI2N2MzZWM5IiwKICAiZ3B1IjogIlRlc2xhIFQ0IiwgInZyYW1fbWliIjogMTUzNjAsICJkcml2ZXIiOiAiNTgwLjE3OC4wNCIsCiAgInB5dGhvbiI6ICIzLjEzLjE1IiwgInRvcmNoIjogIjIuMTEuMCtjdTEyOCIsICJjdWRhX2J1aWxkIjogIjEyLjgiLCAidHJpdG9uIjogIjMuNi4wIiwgIm51bXB5IjogIjIuMy41IiwKICAiY3B1X2FmZmluaXR5IjogWzAsMSwyLDNdLCAiY2dyb3VwX3JhbV9naWIiOiAzMCwgImhvc3RfcmVwb3J0ZWRfcmFtX2dpYiI6IDI5LjM3LAogICJTT1VQX1JFUE8iOiAiL2thZ2dsZS93b3JraW5nL3JlcG8iLCAiU09VUF9SRVNVTFRTIjogIi9rYWdnbGUvd29ya2luZy9yZXN1bHRzIiwKICAib2ZmaWNpYWxfY29tbWl0IjogImM3YzJmY2QxNTk3MmNlOGI4OThkOGQzZmY3MzJlZTE5MmM4MTdjZWIiLAogICJ0cmFpbmluZ190aW1lb3V0X3MiOiA4MCwgInB1YmxpY19xdWFsaXR5X2RlbHRhX2xpbWl0X3BjdCI6IDAuNSwgInBlYWtfcmVncmVzc2lvbl9saW1pdF9wY3QiOiAxMCwKICAiY29uY3VycmVuY3kiOiAxLCAibmV0d29yayI6ICJub25lIiwgImF1dGhlbnRpY2F0ZWRfdXBsb2FkX2F1dG9fc3RhcnRzIjogdHJ1ZSwKICAibm90ZWJvb2tfbGltaXRfYnl0ZXMiOiA4Mzg4NjA4LCAiZXhwb3J0X2xpbWl0X2J5dGVzIjogNjcxMDg4NjQsCiAgImRlZmF1bHRfam9iX3RpbWVvdXRfcyI6IDM2MDAsICJtYXhfam9iX3RpbWVvdXRfcyI6IDE0NDAwLAogICJkZXZpY2VfZXhlY3V0aW9uX3Byb29mIjogZmFsc2UsICJncHVfam9iX3J1bl9ieV91cyI6IGZhbHNlCn0K', 'owned_process_tree.py': 'IyBTaGFyZWQgdGFzayBvcmNoZXN0cmF0aW9uIGZyb20gdGhlIExvUkEgbGFuZTsgbm8ga2VybmVsIG93bmVyc2hpcCBjcmVkaXQuCiIiIkxpbnV4LW9ubHkgZGVkaWNhdGVkLXJ1bm5lciBjbGVhbnVwOyBubyBHUFUgaW1wb3J0cyBvciBiYWNrZ3JvdW5kIGFjdGl2aXR5LgoKT3JpZ2luYWwgdGFzay1vcmNoZXN0cmF0aW9uIGhlbHBlci4gTm90IHBhcnQgb2Ygc29sdXRpb24vIG9yIG93bi1rZXJuZWwgY3JlZGl0LgoiIiIKaW1wb3J0IGN0eXBlcwppbXBvcnQgb3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBzaWduYWwKaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHN5cwppbXBvcnQgdGltZQoKCmRlZiBwcmVwYXJlKCk6CiAgICAiIiJGYWlsIGJlZm9yZSBHUFUgd29yayBpZiByYWNlLXNhZmUgcHJvY2VzcyBvd25lcnNoaXAgaXMgdW5hdmFpbGFibGUuIiIiCiAgICBpZiBub3QgaGFzYXR0cihvcywncGlkZmRfb3BlbicpIG9yIG5vdCBoYXNhdHRyKHNpZ25hbCwncGlkZmRfc2VuZF9zaWduYWwnKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0xpbnV4IHBpZGZkIHByb2Nlc3Mgb3duZXJzaGlwIHJlcXVpcmVkJykKICAgIGZkPW9zLnBpZGZkX29wZW4ob3MuZ2V0cGlkKCkpO29zLmNsb3NlKGZkKQogICAgbGliYz1jdHlwZXMuQ0RMTChOb25lLHVzZV9lcnJubz1UcnVlKQogICAgaWYgbGliYy5wcmN0bCgzNiwxLDAsMCwwKSE9MDoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0Nhbm5vdCBlbmFibGUgY2hpbGQgc3VicmVhcGVyOyBubyBHUFUgam9iIHN0YXJ0ZWQnKQogICAgZW5hYmxlZD1jdHlwZXMuY19pbnQoMCkKICAgIGlmIGxpYmMucHJjdGwoMzcsY3R5cGVzLmJ5cmVmKGVuYWJsZWQpLDAsMCwwKSE9MCBvciBlbmFibGVkLnZhbHVlIT0xOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignQ2hpbGQgc3VicmVhcGVyIHZlcmlmaWNhdGlvbiBmYWlsZWQnKQoKCmRlZiBpZGVudGl0eShwaWQpOgogICAgdHJ5OgogICAgICAgIGZpZWxkcz1QYXRoKGYnL3Byb2Mve3BpZH0vc3RhdCcpLnJlYWRfdGV4dCgpLnJzcGxpdCgnKScsMSlbMV0uc3BsaXQoKQogICAgICAgIHJldHVybiBpbnQoZmllbGRzWzFdKSxpbnQoZmllbGRzWzE5XSksZmllbGRzWzBdCiAgICBleGNlcHQgKE9TRXJyb3IsVmFsdWVFcnJvcixJbmRleEVycm9yKTpyZXR1cm4gTm9uZQoKCmRlZiBkZXNjZW5kYW50cygpOgogICAgcmVjb3Jkcz17fQogICAgZm9yIHAgaW4gUGF0aCgnL3Byb2MnKS5pdGVyZGlyKCk6CiAgICAgICAgaWYgcC5uYW1lLmlzZGlnaXQoKToKICAgICAgICAgICAgcmVjPWlkZW50aXR5KGludChwLm5hbWUpKQogICAgICAgICAgICBpZiByZWMgaXMgbm90IE5vbmU6cmVjb3Jkc1tpbnQocC5uYW1lKV09cmVjCiAgICBvd25lZD17b3MuZ2V0cGlkKCl9CiAgICB3aGlsZSBUcnVlOgogICAgICAgIGFkZD17cGlkIGZvciBwaWQsKHBhcmVudCxfLF8pIGluIHJlY29yZHMuaXRlbXMoKSBpZiBwYXJlbnQgaW4gb3duZWR9LW93bmVkCiAgICAgICAgaWYgbm90IGFkZDpicmVhawogICAgICAgIG93bmVkLnVwZGF0ZShhZGQpCiAgICByZXR1cm4ge3BpZDpyZWNvcmRzW3BpZF0gZm9yIHBpZCBpbiBvd25lZCBpZiBwaWQhPW9zLmdldHBpZCgpIGFuZCBwaWQgaW4gcmVjb3Jkc30KCgpkZWYgc2lnbmFsX2V4YWN0KHBpZCxiaXJ0aCxzaWcpOgogICAgIiIiTmV2ZXIgc2lnbmFsIGEgcmVjeWNsZWQgUElELiBUaGUgRkQgcGlucyBpZGVudGl0eSB0aHJvdWdoIHRoZSBzZW5kLiIiIgogICAgdHJ5OmZkPW9zLnBpZGZkX29wZW4ocGlkKQogICAgZXhjZXB0IFByb2Nlc3NMb29rdXBFcnJvcjpyZXR1cm4KICAgIHRyeToKICAgICAgICBub3c9aWRlbnRpdHkocGlkKQogICAgICAgIGlmIG5vdyBpcyBub3QgTm9uZSBhbmQgbm93WzFdPT1iaXJ0aDoKICAgICAgICAgICAgdHJ5OnNpZ25hbC5waWRmZF9zZW5kX3NpZ25hbChmZCxzaWcsTm9uZSwwKQogICAgICAgICAgICBleGNlcHQgUHJvY2Vzc0xvb2t1cEVycm9yOnBhc3MKICAgIGZpbmFsbHk6b3MuY2xvc2UoZmQpCgoKZGVmIF90cnlfc2lnbmFsKHBpZCxiaXJ0aCxzaWcpOgogICAgdHJ5OnNpZ25hbF9leGFjdChwaWQsYmlydGgsc2lnKQogICAgZXhjZXB0IE9TRXJyb3IgYXMgZXhjOgogICAgICAgICMgRG8gbm90IHVud2luZC9yZWxlYXNlIEdQVSBvd25lcnNoaXAgd2hlbiBjbGVhbnVwIGlzIHVuY2VydGFpbi4KICAgICAgICBxdWFyYW50aW5lKCdDbGVhbnVwIHNpZ25hbCBmYWlsZWQ6ICcrcmVwcihleGMpKQogICAgICAgIHByaW50KCdTT1VQX0NMRUFOVVBfUEVORElORyAnK3JlcHIoZXhjKSxmaWxlPXN5cy5zdGRlcnIsZmx1c2g9VHJ1ZSkKCgpkZWYgY2xlYW51cChwcm9jKToKICAgIHNhdmVkPXtzOnNpZ25hbC5zaWduYWwocyxzaWduYWwuU0lHX0lHTikgZm9yIHMgaW4gKHNpZ25hbC5TSUdJTlQsc2lnbmFsLlNJR1RFUk0pfQogICAgdHJ5OgogICAgICAgIGZyb3plbj1zZXQoKQogICAgICAgIHdoaWxlIFRydWU6CiAgICAgICAgICAgIG9ic2VydmVkPWRlc2NlbmRhbnRzKCkKICAgICAgICAgICAgbmV3PXsocGlkLHZbMV0pIGZvciBwaWQsdiBpbiBvYnNlcnZlZC5pdGVtcygpfS1mcm96ZW4KICAgICAgICAgICAgaWYgbm90IG5ldzpicmVhawogICAgICAgICAgICBmb3IgcGlkLGJpcnRoIGluIG5ldzpfdHJ5X3NpZ25hbChwaWQsYmlydGgsc2lnbmFsLlNJR1NUT1ApCiAgICAgICAgICAgIGZyb3plbi51cGRhdGUobmV3KQogICAgICAgIGZvciBwaWQsYmlydGggaW4gZnJvemVuOl90cnlfc2lnbmFsKHBpZCxiaXJ0aCxzaWduYWwuU0lHS0lMTCkKICAgICAgICBwcm9jLndhaXQoKQogICAgICAgICMgTm8gdGltZW91dC1iYXNlZCByZWxlYXNlOiBpZiBhIEdQVSBjaGlsZCBpcyBzdHVjayBpbiBrZXJuZWwgZXhpdCwKICAgICAgICAjIHJldGFpbiB0aGlzIHN1cGVydmlzb3IgYW5kIGl0cyBsb2NrIHVudGlsIGV4aXQgaXMgYWN0dWFsbHkgb2JzZXJ2ZWQuCiAgICAgICAgd2hpbGUgVHJ1ZToKICAgICAgICAgICAgd2hpbGUgVHJ1ZToKICAgICAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgICAgICBwaWQsXz1vcy53YWl0cGlkKC0xLG9zLldOT0hBTkcpCiAgICAgICAgICAgICAgICAgICAgaWYgbm90IHBpZDpicmVhawogICAgICAgICAgICAgICAgZXhjZXB0IENoaWxkUHJvY2Vzc0Vycm9yOmJyZWFrCiAgICAgICAgICAgIHJlbWFpbmluZz1kZXNjZW5kYW50cygpCiAgICAgICAgICAgIGlmIG5vdCByZW1haW5pbmc6YnJlYWsKICAgICAgICAgICAgZm9yIHBpZCwoXyxiaXJ0aCxfKSBpbiByZW1haW5pbmcuaXRlbXMoKTpfdHJ5X3NpZ25hbChwaWQsYmlydGgsc2lnbmFsLlNJR0tJTEwpCiAgICAgICAgICAgIHRpbWUuc2xlZXAoLjA1KQogICAgZmluYWxseToKICAgICAgICBmb3Igc2lnLGhhbmRsZXIgaW4gc2F2ZWQuaXRlbXMoKTpzaWduYWwuc2lnbmFsKHNpZyxoYW5kbGVyKQoKCmRlZiBydW4oYXJndiwqLGN3ZCxlbnYsc3Rkb3V0LHRpbWVvdXQ9Tm9uZSxzdGRlcnI9c3VicHJvY2Vzcy5TVERPVVQpOgogICAgIiIiUnVuIGluIGRlZGljYXRlZCBzdXBlcnZpc29yIG9ubHk7IGNsZWFudXAgbmVzdGVkIHNlc3Npb25zIG9uIGFueSBmYWlsdXJlLiIiIgogICAgcHJvYz1zdWJwcm9jZXNzLlBvcGVuKGFyZ3YsY3dkPWN3ZCxlbnY9ZW52LHN0ZG91dD1zdGRvdXQsc3RkZXJyPXN0ZGVycixzdGFydF9uZXdfc2Vzc2lvbj1UcnVlKQogICAgdHJ5OgogICAgICAgIGNvZGU9cHJvYy53YWl0KHRpbWVvdXQ9dGltZW91dCkKICAgICAgICBpZiBkZXNjZW5kYW50cygpOgogICAgICAgICAgICBxdWFyYW50aW5lKCdDaGlsZCBleGl0ZWQgbGVhdmluZyBkZXNjZW5kYW50cycpCiAgICAgICAgICAgIGNsZWFudXAocHJvYykKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdDaGlsZCBleGl0ZWQgbGVhdmluZyBkZXNjZW5kYW50czsgY2xlYW5lZCBhbmQgcmVqZWN0ZWQnKQogICAgICAgIHJldHVybiBjb2RlCiAgICBleGNlcHQgQmFzZUV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgcXVhcmFudGluZSgnT3V0ZXIgdGltZW91dC9pbnRlcnJ1cHRpb24vZXJyb3I6ICcrcmVwcihleGMpKQogICAgICAgIGNsZWFudXAocHJvYykKICAgICAgICByYWlzZQoKCmRlZiBpbnN0YWxsX3NpZ25hbF9oYW5kbGVycygpOgogICAgZGVmIGludGVycnVwdGVkKHNpZ251bSxmcmFtZSk6CiAgICAgICAgcXVhcmFudGluZSgnUnVubmVyIGludGVycnVwdGVkIGJ5IHNpZ25hbCAnK3N0cihzaWdudW0pKQogICAgICAgIHJhaXNlIEludGVycnVwdGVkRXJyb3IoJ1J1bm5lciBpbnRlcnJ1cHRlZCBieSBzaWduYWwgJytzdHIoc2lnbnVtKSkKICAgIGZvciBzaWcgaW4gKHNpZ25hbC5TSUdJTlQsc2lnbmFsLlNJR1RFUk0pOnNpZ25hbC5zaWduYWwoc2lnLGludGVycnVwdGVkKQoKIyBBIGNyYXNoLXN1cnZpdmluZyBtYXJrZXIgaXMgcGxhY2VkIGJlZm9yZSBhbnkgR1BVIHN1YnByb2Nlc3MuIE9ubHkgaXRzIGV4YWN0CiMgb3duZXIgbWF5IGNsZWFyIGl0IG9uIGEgY2xlYW4sIG5vbi1pbnRlcnJ1cHRlZCBleGl0IHdpdGggbm8gbGl2ZSBkZXNjZW5kYW50cy4KIyBJbnRlcnJ1cHRlZC90aW1lZC1vdXQgd29yayBsZWF2ZXMgaXQgZm9yIGV4cGxpY2l0IG9wZXJhdG9yIHJlY29uY2lsaWF0aW9uLgpfTUFSS0VSID0gTm9uZQpfVE9LRU4gPSBOb25lCl9LRUVQX01BUktFUiA9IEZhbHNlCgoKZGVmIGNoZWNrX3F1YXJhbnRpbmUocGF0aCk6CiAgICBpZiBQYXRoKHBhdGgpLmV4aXN0cygpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignR1BVMCBxdWFyYW50aW5lZDogcmVjb25jaWxlICcrc3RyKHBhdGgpKycgYmVmb3JlIGFub3RoZXIgcnVuJykKCgpkZWYgYmVnaW5fcXVhcmFudGluZShwYXRoKToKICAgIGdsb2JhbCBfTUFSS0VSLF9UT0tFTixfS0VFUF9NQVJLRVIKICAgIGltcG9ydCBqc29uLHV1aWQKICAgIF9NQVJLRVI9UGF0aChwYXRoKTtfVE9LRU49dXVpZC51dWlkNCgpLmhleDtfS0VFUF9NQVJLRVI9RmFsc2UKICAgIGZkPW9zLm9wZW4oc3RyKF9NQVJLRVIpLG9zLk9fV1JPTkxZfG9zLk9fQ1JFQVR8b3MuT19FWENMfG9zLk9fTk9GT0xMT1csMG82MDApCiAgICB3aXRoIG9zLmZkb3BlbihmZCwndycpIGFzIG91dDoKICAgICAgICBqc29uLmR1bXAoeydvd25lcl90b2tlbic6X1RPS0VOLCdvd25lcl9waWQnOm9zLmdldHBpZCgpLCdzdGF0dXMnOidPV05FRF9HUFVfV09SS19JTl9QUk9HUkVTUyd9LG91dCkKICAgICAgICBvdXQuZmx1c2goKTtvcy5mc3luYyhvdXQuZmlsZW5vKCkpCgoKZGVmIHF1YXJhbnRpbmUocmVhc29uKToKICAgIGdsb2JhbCBfS0VFUF9NQVJLRVIKICAgIF9LRUVQX01BUktFUj1UcnVlCiAgICBpZiBfTUFSS0VSIGlzIE5vbmU6cmV0dXJuCiAgICBpbXBvcnQganNvbgogICAgdHJ5OnZhbHVlPWpzb24ubG9hZHMoX01BUktFUi5yZWFkX3RleHQoKSkKICAgIGV4Y2VwdCAoT1NFcnJvcixWYWx1ZUVycm9yKTpyZXR1cm4KICAgIGlmIHZhbHVlLmdldCgnb3duZXJfdG9rZW4nKSE9X1RPS0VOOnJldHVybgogICAgdmFsdWUudXBkYXRlKHN0YXR1cz0nQkxPQ0tFRF9PUEVSQVRPUl9SRUNPTkNJTElBVElPTl9SRVFVSVJFRCcscmVhc29uPXN0cihyZWFzb24pKQogICAgX01BUktFUi53cml0ZV90ZXh0KGpzb24uZHVtcHModmFsdWUpKydcbicpCgoKZGVmIGZpbmlzaF9xdWFyYW50aW5lKCk6CiAgICBpZiBfTUFSS0VSIGlzIE5vbmUgb3IgX0tFRVBfTUFSS0VSOnJldHVybgogICAgaWYgZGVzY2VuZGFudHMoKToKICAgICAgICBxdWFyYW50aW5lKCdMaXZlIG93bmVkIGRlc2NlbmRhbnRzIHJlbWFpbiBhdCBzdXBlcnZpc29yIGV4aXQnKQogICAgICAgIHJldHVybgogICAgaW1wb3J0IGpzb24KICAgIHRyeTp2YWx1ZT1qc29uLmxvYWRzKF9NQVJLRVIucmVhZF90ZXh0KCkpCiAgICBleGNlcHQgKE9TRXJyb3IsVmFsdWVFcnJvcik6cmV0dXJuCiAgICBpZiB2YWx1ZS5nZXQoJ293bmVyX3Rva2VuJyk9PV9UT0tFTjpfTUFSS0VSLnVubGluaygpCg==', 'runtime_probe.py': 'IiIiUmVhZC1vbmx5LCBzaG9ydC1saXZlZCBkZXBsb3llZC10ZW1wbGF0ZSBnYXRlLiBOZXZlciBwcmludHMgc2VjcmV0IHZhbHVlcy4iIiIKaW1wb3J0IGltcG9ydGxpYi5tZXRhZGF0YSBhcyBtZXRhZGF0YSxqc29uLG9zLHBsYXRmb3JtLHN1YnByb2Nlc3Msc3lzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApFWFBFQ1RFRD17J3RvcmNoJzonMi4xMS4wK2N1MTI4JywndHJpdG9uJzonMy42LjAnLCdudW1weSc6JzIuMy41Jywnc291cC1jbGknOicwLjc3LjAnLCd0cmFuc2Zvcm1lcnMnOic1LjE2LjEnLCdwZWZ0JzonMC4yMC4wJywndHJsJzonMC4yOS4xJywnYml0c2FuZGJ5dGVzJzonMC41MC4yJywnYWNjZWxlcmF0ZSc6JzEuMTQuMCcsJ2RhdGFzZXRzJzonNC44LjUnfQoKZGVmIG1haW4oKToKIG91dD1QYXRoKHN5cy5hcmd2WzFdKTtyPXsncGFzc2VkJzpGYWxzZSwnZ3B1X3dvcmtsb2FkX2V4ZWN1dGVkJzpGYWxzZSwnc2NvcmUnOk5vbmUsJ2FkbWlzc2lvbic6J1VOS05PV04nfQogdHJ5OgogIHZlcnNpb25zPXtrOm1ldGFkYXRhLnZlcnNpb24oaykgZm9yIGsgaW4gRVhQRUNURUR9CiAgci51cGRhdGUocHl0aG9uPXBsYXRmb3JtLnB5dGhvbl92ZXJzaW9uKCkscGFja2FnZXM9dmVyc2lvbnMsdWlkPW9zLmdldHVpZCgpLGNwdV9hZmZpbml0eT1zb3J0ZWQob3Muc2NoZWRfZ2V0YWZmaW5pdHkoMCkpKQogIGFzc2VydCBwbGF0Zm9ybS5weXRob25fdmVyc2lvbigpPT0nMy4xMy4xNScsJ1B5dGhvbiBwaW4gbWlzbWF0Y2gnCiAgYXNzZXJ0IHZlcnNpb25zPT1FWFBFQ1RFRCwnRXhhY3QgcGFja2FnZSBwaW5zIG1pc21hdGNoJwogIGFzc2VydCBvcy5nZXR1aWQoKT09MTAwMDEsJ0RlcGxveWVkIHJ1bnRpbWUgdWlkIG1pc21hdGNoJwogIGFzc2VydCBsZW4ob3Muc2NoZWRfZ2V0YWZmaW5pdHkoMCkpPT00LCdGb3VyIGFzc2lnbmVkIENQVXMgcmVxdWlyZWQnCiAgZm9yIHAsd2FudGVkIGluIFsoJ21lbW9yeS5tYXgnLHN0cigzMCoxMDI0KiozKSksKCdtZW1vcnkuc3dhcC5tYXgnLCcwJyksKCdjcHVzZXQuY3B1cy5lZmZlY3RpdmUnLCcwLTMnKV06CiAgIGFjdHVhbD0oUGF0aCgnL3N5cy9mcy9jZ3JvdXAnKS9wKS5yZWFkX3RleHQoKS5zdHJpcCgpO3JbcF09YWN0dWFsCiAgIGFzc2VydCBhY3R1YWw9PXdhbnRlZCwnRGVwbG95ZWQgY2dyb3VwIG1pc21hdGNoOiAnK3AKICBhc3NlcnQgbm90IFBhdGgoJy92YXIvcnVuL2RvY2tlci5zb2NrJykuZXhpc3RzKCksJ1VuZXhwZWN0ZWQgRG9ja2VyIHNvY2tldCcKICBhc3NlcnQgbm90IFBhdGgoJy9ldGMvc291cC1xdWV1ZS93b3JrZXIuZW52JykuZXhpc3RzKCksJ1VuZXhwZWN0ZWQgd29ya2VyIGVudmlyb25tZW50IGZpbGUnCiAgYXNzZXJ0IG5vdCBhbnkoJ1RPS0VOJyBpbiBrIGFuZCBrIT0nVE9LRU5JWkVSU19QQVJBTExFTElTTScgZm9yIGsgaW4gb3MuZW52aXJvbiksJ1Rva2VuLWJlYXJpbmcgZW52aXJvbm1lbnQgcmVqZWN0ZWQgd2l0aG91dCByZWFkaW5nIGl0cyB2YWx1ZScKICBhc3NlcnQgbm90IG9zLmVudmlyb24uZ2V0KCdUUklUT05fSU5URVJQUkVUJyksJ0ludGVycHJldGVyIG1vZGUgZm9yYmlkZGVuJwogIGltcG9ydCB0b3JjaCx0cml0b24sbnVtcHkKICBhc3NlcnQgdG9yY2guX192ZXJzaW9uX189PScyLjExLjArY3UxMjgnIGFuZCB0b3JjaC52ZXJzaW9uLmN1ZGE9PScxMi44JwogIGFzc2VydCB0cml0b24uX192ZXJzaW9uX189PSczLjYuMCcgYW5kIG51bXB5Ll9fdmVyc2lvbl9fPT0nMi4zLjUnCiAgYXNzZXJ0IHRvcmNoLmN1ZGEuZGV2aWNlX2NvdW50KCk9PTEgYW5kIHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9uYW1lKDApPT0nVGVzbGEgVDQnCiAgYXNzZXJ0IHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9jYXBhYmlsaXR5KDApPT0oNyw1KQogIHIudXBkYXRlKGdwdV9uYW1lPXRvcmNoLmN1ZGEuZ2V0X2RldmljZV9uYW1lKDApLGN1ZGFfYnVpbGQ9dG9yY2gudmVyc2lvbi5jdWRhLHRvcmNoX2dpdD10b3JjaC52ZXJzaW9uLmdpdF92ZXJzaW9uKQogIGZyb20gaHVnZ2luZ2ZhY2VfaHViLmNvbnN0YW50cyBpbXBvcnQgSEZfSFVCX0NBQ0hFCiAgclsncmVzb2x2ZWRfcHVibGljX2h1Yl9jYWNoZSddPXN0cihQYXRoKEhGX0hVQl9DQUNIRSkucmVzb2x2ZSgpKQogIHJbJ3Bhc3NlZCddPVRydWUKIGV4Y2VwdCBCYXNlRXhjZXB0aW9uIGFzIGU6clsnZXJyb3InXT1yZXByKGUpCiBvdXQud3JpdGVfdGV4dChqc29uLmR1bXBzKHIsaW5kZW50PTIpKydcbicpO3ByaW50KGpzb24uZHVtcHMocixpbmRlbnQ9MikpCiByYWlzZSBTeXN0ZW1FeGl0KDAgaWYgclsncGFzc2VkJ10gZWxzZSAxKQppZiBfX25hbWVfXz09J19fbWFpbl9fJzptYWluKCkK', 'prepare_asset_manifest.py': 'IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJSZWFkL2hhc2ggb25seSB0aGUgc2VsZWN0ZWQgcHVibGljIG1vZGVsIGNhY2hlICsgcHVibGljIHRhc2sgZmlsZXM7IG5vIGRvd25sb2FkLiIiIgppbXBvcnQgYXJncGFyc2UsaGFzaGxpYixqc29uLHJlCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApNT0RFTD0nbW9kZWxzLS1Rd2VuLS1Rd2VuMi41LTEuNUItSW5zdHJ1Y3QnClJFVklTSU9OPSc5ODlhYTc5ODBlNGNmODA2ZjgwYzdmZWYyYjFhZGI3YmM3MWFhMzA2JwpEQVRBPXsnZGF0YS90cmFpbi5qc29ubCc6JzkzZDAzNmQ5MTJiZDgxYzMzN2RjMjk4OTNlMTNhNzZmMzYxNjFkOTZmODk2ZWY0ZTAyNDkyZTJmMWEyMWEwZGQnLCdkYXRhL2V2YWxfcHVibGljLmpzb25sJzonYjdmODViNzhiYTMxODZkY2U0ZGVhYjdjZmI4YmY4ZGNmNzk5OGUxNTE3YzU3NDlhYzBmNTAwNTg4NmM2ZjcwZSd9CmRlZiBzaGEocGF0aCk6CiBoPWhhc2hsaWIuc2hhMjU2KCkKIHdpdGggcGF0aC5vcGVuKCdyYicpIGFzIGY6CiAgZm9yIGIgaW4gaXRlcihsYW1iZGE6Zi5yZWFkKDQqMTAyNCoxMDI0KSxiJycpOmgudXBkYXRlKGIpCiByZXR1cm4gaC5oZXhkaWdlc3QoKQpkZWYgYnVpbGQoY2FjaGUscmVwbyxyZXZpc2lvbj1SRVZJU0lPTik6CiBjYWNoZT1QYXRoKGNhY2hlKS5yZXNvbHZlKCk7cmVwbz1QYXRoKHJlcG8pLnJlc29sdmUoKQogaWYgbm90IHJlLmZ1bGxtYXRjaCgnWzAtOWEtZl17NDB9JyxyZXZpc2lvbik6cmFpc2UgVmFsdWVFcnJvcignUmV2aXNpb24gbXVzdCBiZTQwaGV4JykKIHJvb3Q9Y2FjaGUvTU9ERUwKIGlmIHJvb3QuaXNfc3ltbGluaygpIG9yIG5vdCByb290LmlzX2RpcigpOnJhaXNlIFZhbHVlRXJyb3IoJ1NlbGVjdGVkIG1vZGVsIGNhY2hlIG1pc3Npbmcgb3Igc3ltbGlua2VkJykKIGZvciBkaXJlY3RvcnkgaW4gKHJvb3QvJ3JlZnMnLHJvb3QvJ3NuYXBzaG90cycscm9vdC8nYmxvYnMnKToKICBpZiBkaXJlY3RvcnkuaXNfc3ltbGluaygpIG9yIG5vdCBkaXJlY3RvcnkuaXNfZGlyKCk6cmFpc2UgVmFsdWVFcnJvcignTW9kZWwgbWV0YWRhdGEvYmxvYiBkaXJlY3RvcmllcyBtdXN0IGJlIHJlYWwgbG9jYWwgZGlyZWN0b3JpZXMnKQogcmVmPXJvb3QvJ3JlZnMvbWFpbicKIGlmIHJlZi5pc19zeW1saW5rKCkgb3Igbm90IHJlZi5pc19maWxlKCkgb3IgcmVmLnJlYWRfdGV4dCgpLnN0cmlwKCkhPXJldmlzaW9uOnJhaXNlIFZhbHVlRXJyb3IoJ3JlZnMvbWFpbiBtdXN0IGlkZW50aWZ5IGV4YWN0IHBpbm5lZCByZXZpc2lvbjsgbm8gc2lsZW50IHJldmlzaW9uIHN1YnN0aXR1dGlvbicpCiBzbmFwc2hvdD1yb290LydzbmFwc2hvdHMnL3JldmlzaW9uCiBpZiBzbmFwc2hvdC5pc19zeW1saW5rKCkgb3Igbm90IHNuYXBzaG90LmlzX2RpcigpOnJhaXNlIFZhbHVlRXJyb3IoJ1NuYXBzaG90IGRpcmVjdG9yeSBhYnNlbnQ7IHJlc3RvcmUgdmFsaWRhdGVkIHJldGFpbmVkIGxpbmtzIHNlcGFyYXRlbHksIGRvIG5vdCBndWVzcycpCiBmaWxlcz17c3RyKHJlZi5yZWxhdGl2ZV90byhjYWNoZSkpOnNoYShyZWYpfTttZW1vPXt9CiBmb3IgcCBpbiBzb3J0ZWQoc25hcHNob3Qucmdsb2IoJyonKSk6CiAgaWYgcC5pc19kaXIoKToKICAgaWYgcC5pc19zeW1saW5rKCk6cmFpc2UgVmFsdWVFcnJvcignTGlua2VkIHNuYXBzaG90IGRpcmVjdG9yeSByZWplY3RlZCcpCiAgIGNvbnRpbnVlCiAgcmVzb2x2ZWQ9cC5yZXNvbHZlKHN0cmljdD1UcnVlKQogIGlmIG5vdCByZXNvbHZlZC5pc19yZWxhdGl2ZV90bygocm9vdC8nYmxvYnMnKS5yZXNvbHZlKCkpIG9yIG5vdCByZXNvbHZlZC5pc19maWxlKCk6cmFpc2UgVmFsdWVFcnJvcignU25hcHNob3QgdGFyZ2V0IG11c3QgcmVtYWluIGluc2lkZSBzZWxlY3RlZCBtb2RlbCBibG9icycpCiAgaWYgcmVzb2x2ZWQgbm90IGluIG1lbW86bWVtb1tyZXNvbHZlZF09c2hhKHJlc29sdmVkKQogIGZpbGVzW3N0cihwLnJlbGF0aXZlX3RvKGNhY2hlKSldPW1lbW9bcmVzb2x2ZWRdCiBpZiBsZW4oZmlsZXMpPDI6cmFpc2UgVmFsdWVFcnJvcignRW1wdHkgbW9kZWwgc25hcHNob3QnKQogZGF0YT17fQogZm9yIG5hbWUsd2FudGVkIGluIERBVEEuaXRlbXMoKToKICBwPXJlcG8vbmFtZQogIGlmIG5vdCBwLmlzX2ZpbGUoKSBvciBwLmlzX3N5bWxpbmsoKSBvciBub3QgcC5yZXNvbHZlKCkuaXNfcmVsYXRpdmVfdG8ocmVwbykgb3Igc2hhKHApIT13YW50ZWQ6cmFpc2UgVmFsdWVFcnJvcignUHVibGljIGRhdGEgaGFzaCBtaXNtYXRjaDogJytuYW1lKQogIGRhdGFbbmFtZV09d2FudGVkCiByZXR1cm4geydtb2RlbF9pZCc6J1F3ZW4vUXdlbjIuNS0xLjVCLUluc3RydWN0JywncmV2aXNpb24nOnJldmlzaW9uLCdmaWxlcyc6ZmlsZXN9LGRhdGEKaWYgX19uYW1lX189PSdfX21haW5fXyc6CiBwPWFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPV9fZG9jX18pO3AuYWRkX2FyZ3VtZW50KCctLWNhY2hlJyxyZXF1aXJlZD1UcnVlKTtwLmFkZF9hcmd1bWVudCgnLS1yZXBvJyxyZXF1aXJlZD1UcnVlKTtwLmFkZF9hcmd1bWVudCgnLS1vdXQnLHJlcXVpcmVkPVRydWUpO2E9cC5wYXJzZV9hcmdzKCk7bWFuaWZlc3QsZGF0YT1idWlsZChhLmNhY2hlLGEucmVwbyk7b3V0PVBhdGgoYS5vdXQpCiB3aXRoIG91dC5vcGVuKCd4JykgYXMgZjpqc29uLmR1bXAobWFuaWZlc3QsZixpbmRlbnQ9Mik7Zi53cml0ZSgnXG4nKQogcHJpbnQoanNvbi5kdW1wcyh7J21vZGVsX2NhY2hlX21hbmlmZXN0JzpzdHIob3V0KSwnbW9kZWxfY2FjaGVfbWFuaWZlc3Rfc2hhMjU2JzpzaGEob3V0KSwnZGF0YV9maWxlcyc6ZGF0YX0saW5kZW50PTIpKQo=', 'process_guard.py': 'IiIiSXNvbGF0ZWQgTGludXggc3VicmVhcGVyIGd1YXJkaWFuLiBPd25zIGNoaWxkcmVuIGFuZCBob2xkcyB0aGUgR1BVIGxvY2suCgpUaGUgZ3VhcmRpYW4gd2F0Y2hlcyBhIHBhcmVudC1saXZlbmVzcyBwaXBlOiBkcml2ZXIgU0lHS0lMTCBvciBvdGhlciBhYnJ1cHQKZGVhdGggY2xvc2VzIHRoZSBwaXBlIGFuZCB0cmlnZ2VycyBxdWFyYW50aW5lZCBjbGVhbnVwLiBQSURGRCBzaWduYWxpbmcgaW4gdGhlCnNoYXJlZCB0YXNrIGhlbHBlciBwcmV2ZW50cyBQSUQgcmV1c2UgcmFjZXM7IGRlc2NlbmRhbnRzIGFyZSBvbmx5IHRoaXMgZGVkaWNhdGVkCmd1YXJkaWFuJ3MgY2hpbGRyZW4sIGluY2x1ZGluZyBuZXdseSBhZG9wdGVkIG9ycGhhbnMgZnJvbSBuZXN0ZWQgc2Vzc2lvbnMuClRoaXMgaXMgdGFzayBvcmNoZXN0cmF0aW9uLCBuZXZlciBzb2x1dGlvbi8gd29yayBvciBrZXJuZWwgcGFydGljaXBhdGlvbi4KIiIiCmltcG9ydCBqc29uCmltcG9ydCBvcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IHNlbGVjdAppbXBvcnQgc2lnbmFsCmltcG9ydCBzdWJwcm9jZXNzCmltcG9ydCBzeXMKaW1wb3J0IHRpbWUKCmltcG9ydCBvd25lZF9wcm9jZXNzX3RyZWUgYXMgdHJlZQoKCmRlZiBxdWFyYW50aW5lKHBhdGgsIHJlYXNvbik6CiAgICAiIiJGaXJzdCBmYWlsdXJlIHdpbnM7IG5ldmVyIG92ZXJ3cml0ZSBvciBhdXRvbWF0aWNhbGx5IGNsZWFyIGEgbWFya2VyLiIiIgogICAgdHJ5OgogICAgICAgIGRlc2NyaXB0b3IgPSBvcy5vcGVuKHBhdGgsIG9zLk9fV1JPTkxZIHwgb3MuT19DUkVBVCB8IG9zLk9fRVhDTCB8IG9zLk9fTk9GT0xMT1csIDBvNjAwKQogICAgZXhjZXB0IEZpbGVFeGlzdHNFcnJvcjoKICAgICAgICByZXR1cm4KICAgIHdpdGggb3MuZmRvcGVuKGRlc2NyaXB0b3IsICJ3IikgYXMgc3RyZWFtOgogICAgICAgIGpzb24uZHVtcCh7InN0YXR1cyI6ICJPUEVSQVRPUl9SRUNPTkNJTElBVElPTl9SRVFVSVJFRCIsICJyZWFzb24iOiByZWFzb24sCiAgICAgICAgICAgICAgICAgICAiZ3VhcmRpYW5fcGlkIjogb3MuZ2V0cGlkKCksICJjcmVhdGVkX3VuaXgiOiB0aW1lLnRpbWUoKX0sIHN0cmVhbSkKICAgICAgICBzdHJlYW0ud3JpdGUoIlxuIikKICAgICAgICBzdHJlYW0uZmx1c2goKQogICAgICAgIG9zLmZzeW5jKHN0cmVhbS5maWxlbm8oKSkKCgpkZWYgX3dyaXRlX3JlY2VpcHQocGF0aCwgdmFsdWUpOgogICAgcGF0aCA9IFBhdGgocGF0aCkKICAgIHBlbmRpbmcgPSBwYXRoLndpdGhfbmFtZShwYXRoLm5hbWUgKyAiLnBlbmRpbmciKQogICAgcGVuZGluZy53cml0ZV90ZXh0KGpzb24uZHVtcHModmFsdWUsIGluZGVudD0yKSArICJcbiIpCiAgICBwZW5kaW5nLnJlcGxhY2UocGF0aCkKCgpkZWYgZ3VhcmRpYW5fbWFpbigpOgogICAgY29uZmlnID0ganNvbi5sb2FkcyhzeXMuc3RkaW4ucmVhZGxpbmUoKSkKICAgIHJlYWRfZmQsIGxvY2tfZmQgPSBjb25maWdbInBhcmVudF9waXBlX2ZkIl0sIGNvbmZpZ1sibG9ja19mZCJdCiAgICBzdGFydGVkLCByZXF1ZXN0ZWRfc2lnbmFsLCBwcm9jZXNzID0gdGltZS5tb25vdG9uaWMoKSwgW05vbmVdLCBOb25lCiAgICByZXN1bHQgPSB7ImV4aXRfY29kZSI6IDEyNSwgInJvb3RfZXhpdF9jb2RlIjogTm9uZSwgInRpbWVkX291dCI6IEZhbHNlLAogICAgICAgICAgICAgICJpbnRlcnJ1cHRlZCI6IEZhbHNlLCAiY2xlYW51cF9jb25maXJtZWQiOiBGYWxzZSwKICAgICAgICAgICAgICAicmVhc29uIjogTm9uZSwgImd1YXJkaWFuX3BpZCI6IG9zLmdldHBpZCgpfQogICAgZGVmIHJlcXVlc3Qoc2lnbnVtLCBmcmFtZSk6CiAgICAgICAgcmVxdWVzdGVkX3NpZ25hbFswXSA9IHNpZ251bQogICAgZm9yIHNpZyBpbiAoc2lnbmFsLlNJR0lOVCwgc2lnbmFsLlNJR1RFUk0pOgogICAgICAgIHNpZ25hbC5zaWduYWwoc2lnLCByZXF1ZXN0KQogICAgdHJ5OgogICAgICAgIHRyZWUucHJlcGFyZSgpICAjIHJlcXVpcmVkIGJlZm9yZSBhbnkgY2hpbGQ7IG5ldmVyIHNpbGVudGx5IGRlZ3JhZGUKICAgICAgICB3aXRoIFBhdGgoY29uZmlnWyJsb2ciXSkub3BlbigieCIpIGFzIHN0cmVhbToKICAgICAgICAgICAgcHJvY2VzcyA9IHN1YnByb2Nlc3MuUG9wZW4oY29uZmlnWyJjb21tYW5kIl0sIGN3ZD1jb25maWdbImN3ZCJdLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBzdGRvdXQ9c3RyZWFtLCBzdGRlcnI9c3VicHJvY2Vzcy5TVERPVVQsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHN0YXJ0X25ld19zZXNzaW9uPVRydWUpCiAgICAgICAgICAgIHdoaWxlIFRydWU6CiAgICAgICAgICAgICAgICByZWFkeSwgXywgXyA9IHNlbGVjdC5zZWxlY3QoW3JlYWRfZmRdLCBbXSwgW10sIDAuMDUpCiAgICAgICAgICAgICAgICBpZiByZWFkeSBhbmQgb3MucmVhZChyZWFkX2ZkLCAxKSA9PSBiIiI6CiAgICAgICAgICAgICAgICAgICAgcmVzdWx0LnVwZGF0ZShyZWFzb249InBhcmVudF9saXZlbmVzc19waXBlX2Nsb3NlZCIsIGludGVycnVwdGVkPVRydWUpCiAgICAgICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgICAgIGlmIHJlcXVlc3RlZF9zaWduYWxbMF0gaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICAgICAgcmVzdWx0LnVwZGF0ZShyZWFzb249Imd1YXJkaWFuX3NpZ25hbF8iICsgc3RyKHJlcXVlc3RlZF9zaWduYWxbMF0pLCBpbnRlcnJ1cHRlZD1UcnVlKQogICAgICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgICAgICBpZiB0aW1lLm1vbm90b25pYygpIC0gc3RhcnRlZCA+PSBjb25maWdbInRpbWVvdXQiXToKICAgICAgICAgICAgICAgICAgICByZXN1bHQudXBkYXRlKHJlYXNvbj0ib3V0ZXJfdGltZW91dCIsIHRpbWVkX291dD1UcnVlKQogICAgICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgICAgICByZXR1cm5jb2RlID0gcHJvY2Vzcy5wb2xsKCkKICAgICAgICAgICAgICAgIGlmIHJldHVybmNvZGUgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICAgICAgcmVzdWx0WyJyb290X2V4aXRfY29kZSJdID0gcmV0dXJuY29kZQogICAgICAgICAgICAgICAgICAgIGlmIHRyZWUuZGVzY2VuZGFudHMoKToKICAgICAgICAgICAgICAgICAgICAgICAgcmVzdWx0WyJyZWFzb24iXSA9ICJyb290X2V4aXRlZF93aXRoX293bmVkX2Rlc2NlbmRhbnRzIgogICAgICAgICAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICAgICAgICAgIHJlc3VsdC51cGRhdGUoZXhpdF9jb2RlPXJldHVybmNvZGUsIGNsZWFudXBfY29uZmlybWVkPVRydWUpCiAgICAgICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgaWYgcmVzdWx0WyJyZWFzb24iXSBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgIHF1YXJhbnRpbmUoY29uZmlnWyJtYXJrZXIiXSwgcmVzdWx0WyJyZWFzb24iXSkKICAgICAgICAgICAgICAgICMgY2xlYW51cCBpZ25vcmVzIHJlcGVhdGVkIElOVC9URVJNLCBzdG9wcy9yZXNjYW5zL2tpbGxzIGJ5CiAgICAgICAgICAgICAgICAjIHBpZGZkIGFuZCB3YWl0cy9yZWFwcyB3aXRob3V0IGFueSB0aW1lb3V0LXVubG9jayBlc2NhcGUuCiAgICAgICAgICAgICAgICB0cmVlLmNsZWFudXAocHJvY2VzcykKICAgICAgICAgICAgICAgIHJlc3VsdC51cGRhdGUocm9vdF9leGl0X2NvZGU9cHJvY2Vzcy5yZXR1cm5jb2RlLCBjbGVhbnVwX2NvbmZpcm1lZD1UcnVlKQogICAgICAgICAgICAgICAgaWYgcmVzdWx0WyJ0aW1lZF9vdXQiXToKICAgICAgICAgICAgICAgICAgICByZXN1bHRbImV4aXRfY29kZSJdID0gMTI0CiAgICAgICAgICAgICAgICBlbGlmIHJlc3VsdFsiaW50ZXJydXB0ZWQiXToKICAgICAgICAgICAgICAgICAgICByZXN1bHRbImV4aXRfY29kZSJdID0gMTMwCiAgICBleGNlcHQgQmFzZUV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgcmVzdWx0LnVwZGF0ZShyZWFzb249Imd1YXJkaWFuX2V4Y2VwdGlvbjoiICsgcmVwcihleGMpLCBleGl0X2NvZGU9MTI1KQogICAgICAgIHRyeToKICAgICAgICAgICAgcXVhcmFudGluZShjb25maWdbIm1hcmtlciJdLCByZXN1bHRbInJlYXNvbiJdKQogICAgICAgIGZpbmFsbHk6CiAgICAgICAgICAgIGlmIHByb2Nlc3MgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICB0cmVlLmNsZWFudXAocHJvY2VzcykKICAgICAgICAgICAgcmVzdWx0WyJjbGVhbnVwX2NvbmZpcm1lZCJdID0gbm90IHRyZWUuZGVzY2VuZGFudHMoKQogICAgZmluYWxseToKICAgICAgICByZXN1bHRbIm91dGVyX3dhbGxfcyJdID0gdGltZS5tb25vdG9uaWMoKSAtIHN0YXJ0ZWQKICAgICAgICBfd3JpdGVfcmVjZWlwdChjb25maWdbInJlY2VpcHQiXSwgcmVzdWx0KQogICAgICAgIG9zLmNsb3NlKHJlYWRfZmQpCiAgICAgICAgIyBUaGUgaW5oZXJpdGVkIG9wZW4tZmlsZSBkZXNjcmlwdGlvbiByZXRhaW5zIHRoZSBjYW5vbmljYWwgZmxvY2sgZXZlbgogICAgICAgICMgaWYgdGhlIGRyaXZlciBoYXMgZGllZC4gQ2xvc2UgT05MWSBhZnRlciBvd25lZC10cmVlIGNsZWFudXAgY29tcGxldGVzLgogICAgICAgIG9zLmNsb3NlKGxvY2tfZmQpCiAgICByZXR1cm4gcmVzdWx0WyJleGl0X2NvZGUiXQoKCmRlZiBydW4oY29tbWFuZCwgKiwgY3dkLCBlbnYsIGxvZywgdGltZW91dCwgbG9ja19mZCwKICAgICAgICBtYXJrZXI9Ii90bXAvc291cC1ncHUwLmxvY2suYmxvY2tlZCIpOgogICAgIiIiRGVkaWNhdGVkIGRyaXZlciBBUEkuIEFueSBCYXNlRXhjZXB0aW9uIGFza3MgZ3VhcmRpYW4gY2xlYW51cCBhbmQgd2FpdHMuIiIiCiAgICByZWNlaXB0ID0gUGF0aChzdHIobG9nKSArICIucHJvY2Vzcy5qc29uIikucmVzb2x2ZSgpCiAgICBpZiByZWNlaXB0LmV4aXN0cygpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiUHJvY2VzcyByZWNlaXB0IGFscmVhZHkgZXhpc3RzOyByZWZ1c2UgdG8gb3ZlcndyaXRlIikKICAgIHJlYWRfZmQsIHdyaXRlX2ZkID0gb3MucGlwZSgpCiAgICBjb25maWcgPSB7ImNvbW1hbmQiOiBbc3RyKHgpIGZvciB4IGluIGNvbW1hbmRdLCAiY3dkIjogc3RyKFBhdGgoY3dkKS5yZXNvbHZlKCkpLAogICAgICAgICAgICAgICJsb2ciOiBzdHIoUGF0aChsb2cpLnJlc29sdmUoKSksICJ0aW1lb3V0IjogdGltZW91dCwKICAgICAgICAgICAgICAicmVjZWlwdCI6IHN0cihyZWNlaXB0KSwgInBhcmVudF9waXBlX2ZkIjogcmVhZF9mZCwKICAgICAgICAgICAgICAibG9ja19mZCI6IGxvY2tfZmQsICJtYXJrZXIiOiBzdHIobWFya2VyKX0KICAgIGd1YXJkaWFuID0gTm9uZQogICAgaW50ZXJydXB0ZWQgPSBOb25lCiAgICBkZWYgcmVxdWVzdChzaWdudW0sIGZyYW1lKToKICAgICAgICByYWlzZSBJbnRlcnJ1cHRlZEVycm9yKCJEcml2ZXIgcmVjZWl2ZWQgc2lnbmFsICIgKyBzdHIoc2lnbnVtKSkKICAgIG9sZF9oYW5kbGVycyA9IHtzaWc6IHNpZ25hbC5zaWduYWwoc2lnLCByZXF1ZXN0KSBmb3Igc2lnIGluIChzaWduYWwuU0lHSU5ULCBzaWduYWwuU0lHVEVSTSl9CiAgICB0cnk6CiAgICAgICAgZ3VhcmRpYW4gPSBzdWJwcm9jZXNzLlBvcGVuKFtzeXMuZXhlY3V0YWJsZSwgc3RyKFBhdGgoX19maWxlX18pLnJlc29sdmUoKSksICItLWd1YXJkaWFuIl0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHN0ZGluPXN1YnByb2Nlc3MuUElQRSwgZW52PWVudiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgcGFzc19mZHM9KHJlYWRfZmQsIGxvY2tfZmQpLCBzdGFydF9uZXdfc2Vzc2lvbj1UcnVlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB0ZXh0PVRydWUpCiAgICAgICAgb3MuY2xvc2UocmVhZF9mZCkKICAgICAgICByZWFkX2ZkID0gTm9uZQogICAgICAgIGd1YXJkaWFuLnN0ZGluLndyaXRlKGpzb24uZHVtcHMoY29uZmlnKSArICJcbiIpCiAgICAgICAgZ3VhcmRpYW4uc3RkaW4uY2xvc2UoKQogICAgICAgIGd1YXJkaWFuLndhaXQoKQogICAgZXhjZXB0IEJhc2VFeGNlcHRpb24gYXMgZXhjOgogICAgICAgIGludGVycnVwdGVkID0gcmVwcihleGMpCiAgICAgICAgIyBJZ25vcmUgcmVwZWF0ZWQgc3RvcCBzaWduYWxzIHVudGlsIHRoZSBndWFyZGlhbiBjb25maXJtcyBjbGVhbnVwLgogICAgICAgIGZvciBzaWcgaW4gb2xkX2hhbmRsZXJzOgogICAgICAgICAgICBzaWduYWwuc2lnbmFsKHNpZywgc2lnbmFsLlNJR19JR04pCiAgICAgICAgaWYgd3JpdGVfZmQgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIG9zLmNsb3NlKHdyaXRlX2ZkKQogICAgICAgICAgICB3cml0ZV9mZCA9IE5vbmUKICAgICAgICBpZiBndWFyZGlhbiBpcyBub3QgTm9uZToKICAgICAgICAgICAgaWYgZ3VhcmRpYW4uc3RkaW4gaXMgbm90IE5vbmUgYW5kIG5vdCBndWFyZGlhbi5zdGRpbi5jbG9zZWQ6CiAgICAgICAgICAgICAgICBndWFyZGlhbi5zdGRpbi5jbG9zZSgpCiAgICAgICAgICAgIGd1YXJkaWFuLndhaXQoKQogICAgZmluYWxseToKICAgICAgICBpZiByZWFkX2ZkIGlzIG5vdCBOb25lOgogICAgICAgICAgICBvcy5jbG9zZShyZWFkX2ZkKQogICAgICAgIGlmIHdyaXRlX2ZkIGlzIG5vdCBOb25lOgogICAgICAgICAgICBvcy5jbG9zZSh3cml0ZV9mZCkKICAgICAgICBmb3Igc2lnLCBoYW5kbGVyIGluIG9sZF9oYW5kbGVycy5pdGVtcygpOgogICAgICAgICAgICBzaWduYWwuc2lnbmFsKHNpZywgaGFuZGxlcikKICAgIGlmIG5vdCByZWNlaXB0LmlzX2ZpbGUoKToKICAgICAgICBxdWFyYW50aW5lKG1hcmtlciwgImd1YXJkaWFuX21pc3NpbmdfcmVjZWlwdCIpCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJHdWFyZGlhbiByZWNlaXB0IG1pc3Npbmc7IEdQVSBxdWFyYW50aW5lIHJlcXVpcmVzIG9wZXJhdG9yIHJlY29uY2lsaWF0aW9uIikKICAgIHJlc3VsdCA9IGpzb24ubG9hZHMocmVjZWlwdC5yZWFkX3RleHQoKSkKICAgIGlmIGludGVycnVwdGVkIGlzIG5vdCBOb25lOgogICAgICAgIHJlc3VsdC51cGRhdGUocGFyZW50X2V4Y2VwdGlvbj1pbnRlcnJ1cHRlZCwgaW50ZXJydXB0ZWQ9VHJ1ZSkKICAgICAgICBpZiByZXN1bHRbImV4aXRfY29kZSJdID09IDA6CiAgICAgICAgICAgIHJlc3VsdFsiZXhpdF9jb2RlIl0gPSAxMzAKICAgICAgICBxdWFyYW50aW5lKG1hcmtlciwgImRyaXZlcl9pbnRlcnJ1cHRpb246IiArIGludGVycnVwdGVkKQogICAgaWYgbm90IHJlc3VsdFsiY2xlYW51cF9jb25maXJtZWQiXToKICAgICAgICBxdWFyYW50aW5lKG1hcmtlciwgImd1YXJkaWFuX3VuY2VydGFpbl9jbGVhbnVwIikKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkd1YXJkaWFuIGNsZWFudXAgdW5jZXJ0YWluOyBxdWFyYW50aW5lIHJldGFpbmVkIikKICAgIHJldHVybiByZXN1bHQKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgaWYgc3lzLmFyZ3ZbMTpdICE9IFsiLS1ndWFyZGlhbiJdOgogICAgICAgIHJhaXNlIFN5c3RlbUV4aXQoIkludGVybmFsIGd1YXJkaWFuIGVudHJ5cG9pbnQgb25seSIpCiAgICByYWlzZSBTeXN0ZW1FeGl0KGd1YXJkaWFuX21haW4oKSkK', 'evidence_reader.py': 'IiIiUHJvcG9zZWQgbmV4dC12ZXJzaW9uIGxvZyByZWFkZXI7IGRvZXMgbm90IGFsdGVyIGFueSBydW4vY29tcGxldGlvbiBkZWNpc2lvbi4KCkFjY2VwdCBhIHN0cnVjdHVyZWQgSlNPTiBvYmplY3Qgb25seSBhZnRlciBhbiBleGFjdCBsaW5lLWFuY2hvcmVkIGV2ZW50IHByZWZpeCwKaWdub3JpbmcgdGVybWluYWwgQU5TSSBlc2NhcGUgZm9ybWF0dGluZyBhbmQgSlNPTiB3aGl0ZXNwYWNlIGxpbmUgd3JhcHBpbmcuCk1hbGZvcm1lZC90cnVuY2F0ZWQvb3ZlcnNpemUvZHVwbGljYXRlLWtleS9ub25maW5pdGUgb2JqZWN0cyBmYWlsIGNsb3NlZC4KIiIiCmltcG9ydCBqc29uCmltcG9ydCByZQoKIyBDU0ksIE9TQyAoQkVMIG9yIFNUIHRlcm1pbmF0aW9uKSwgYW5kIHN0YW5kYXJkIHR3by1jaGFyYWN0ZXIgQU5TSSBlc2NhcGVzLgpfQU5TSSA9IHJlLmNvbXBpbGUocidceDFiXF1bXlx4MDdceDFiXSooPzpceDA3fFx4MWJcXCl8XHgxYlxbWzAtP10qWyAtL10qW0Atfl18XHgxYltALV9dJykKCmRlZiBfcmVqZWN0KHZhbHVlKToKICAgIHJhaXNlIFZhbHVlRXJyb3IoJ05vbmZpbml0ZSBKU09OIGNvbnN0YW50OiAnICsgdmFsdWUpCgoKZGVmIF91bmlxdWVfcGFpcnMocGFpcnMpOgogICAgcmVzdWx0ID0ge30KICAgIGZvciBrZXksIHZhbHVlIGluIHBhaXJzOgogICAgICAgIGlmIGtleSBpbiByZXN1bHQ6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0R1cGxpY2F0ZSBKU09OIGtleTogJyArIGtleSkKICAgICAgICByZXN1bHRba2V5XSA9IHZhbHVlCiAgICByZXR1cm4gcmVzdWx0CgoKZGVmIGV2ZW50cyh0ZXh0LCBwcmVmaXgsICosIG1heF9ldmVudF9jaGFycz0zMjc2OCk6CiAgICAiIiJZaWVsZCB1bmFtYmlndW91cyBkaWN0IGV2ZW50czsgbmV2ZXIgaW5mZXIgZnVsbC1ydW4gb3IgQ1VEQSBjb21wbGV0aW9uLiIiIgogICAgaWYgbm90IGlzaW5zdGFuY2UocHJlZml4LCBzdHIpIG9yIG5vdCBwcmVmaXggb3IgJ1xuJyBpbiBwcmVmaXggb3IgJ1xyJyBpbiBwcmVmaXg6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignQSBub25lbXB0eSBzaW5nbGUtbGluZSBleGFjdCBwcmVmaXggaXMgcmVxdWlyZWQnKQogICAgY2xlYW4gPSBfQU5TSS5zdWIoJycsIHRleHQpLnJlcGxhY2UoJ1xyXG4nLCAnXG4nKS5yZXBsYWNlKCdccicsICdcbicpCiAgICBkZWNvZGVyID0ganNvbi5KU09ORGVjb2RlcihvYmplY3RfcGFpcnNfaG9vaz1fdW5pcXVlX3BhaXJzLCBwYXJzZV9jb25zdGFudD1fcmVqZWN0KQogICAgZm9yIG1hdGNoIGluIHJlLmZpbmRpdGVyKHInKD9tKV5bIFx0XSonICsgcmUuZXNjYXBlKHByZWZpeCksIGNsZWFuKToKICAgICAgICBmcmFnbWVudCA9IGNsZWFuW21hdGNoLmVuZCgpOm1hdGNoLmVuZCgpICsgbWF4X2V2ZW50X2NoYXJzXS5sc3RyaXAoKQogICAgICAgIHRyeToKICAgICAgICAgICAgcmVjb3JkLCBlbmQgPSBkZWNvZGVyLnJhd19kZWNvZGUoZnJhZ21lbnQpCiAgICAgICAgZXhjZXB0IChWYWx1ZUVycm9yLCBUeXBlRXJyb3IsIFJlY3Vyc2lvbkVycm9yKToKICAgICAgICAgICAgY29udGludWUKICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShyZWNvcmQsIGRpY3QpOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGlmIGZyYWdtZW50W2VuZDpdLnNwbGl0KCdcbicsIDEpWzBdLnN0cmlwKCk6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgeWllbGQgcmVjb3JkCg==', 'web_runner.py': '...', 'candidate_profile.json': 'ewogICJzdHJlYW1lZF9tb21lbnRfZWFnZXJfb2ZmbG9hZF92MSI6IHsKICAgICJtb2RlIjogInN0cmVhbWVkX21vbWVudF9lYWdlcl9vZmZsb2FkX3YxIiwKICAgICJ0aXRsZSI6ICJFeHBlcmltZW50YWwgc3RyZWFtZWQgbW9tZW50cyArIGVhZ2VyIGV4YWN0IHN0b3JhZ2UgKyBzYXZlZCBhY3RpdmF0aW9uIG9mZmxvYWQiLAogICAgInByZWZsaWdodCI6ICJzb2x1dGlvbi5vZmZsb2FkX3ByZWZsaWdodCIsCiAgICAicHJlZmxpZ2h0X3N0YXR1cyI6ICJTVFJVQ1RVUkFMX0NIRUNLU19DT01QTEVURV9OVU1FUklDQUxfUkVTVUxUU19SRUNPUkRFRCIsCiAgICAia2VybmVscyI6IFsKICAgICAgInNvdXBfY2VfbW9tZW50X3RpbGVzIiwKICAgICAgInNvdXBfY2VfbW9tZW50X21lcmdlIiwKICAgICAgInNvdXBfbmY0X2RlcXVhbnRpemUiLAogICAgICAic291cF9ybXNub3JtX2ZvcndhcmQiLAogICAgICAic291cF9ybXNub3JtX2JhY2t3YXJkIgogICAgXSwKICAgICJyb3V0ZXMiOiBbCiAgICAgIHsKICAgICAgICAicHJlZml4IjogIlNPVVBfU1RSRUFNRURfQ0UgIiwKICAgICAgICAiZXZlbnQiOiAib3duZWRfY29uc3VtZXJfZ3VhcmQiLAogICAgICAgICJwb3NpdGl2ZV9maWVsZCI6ICJjb3VudCIsCiAgICAgICAgIm1hdGNoIjogewogICAgICAgICAgImNvdW50IjogNjQKICAgICAgICB9CiAgICAgIH0sCiAgICAgIHsKICAgICAgICAicHJlZml4IjogIlNPVVBfU1RSRUFNRURfQ0UgIiwKICAgICAgICAiZXZlbnQiOiAiYmFja2JvbmVfb25jZSIsCiAgICAgICAgInBvc2l0aXZlX2ZpZWxkIjogImNvdW50IiwKICAgICAgICAibWF0Y2giOiB7CiAgICAgICAgICAiY291bnQiOiA2NAogICAgICAgIH0KICAgICAgfSwKICAgICAgewogICAgICAgICJwcmVmaXgiOiAiU09VUF9TVFJFQU1FRF9DRSAiLAogICAgICAgICJldmVudCI6ICJzdW1tYXJ5X2xvc3NfY29tcGxldGVkIiwKICAgICAgICAicG9zaXRpdmVfZmllbGQiOiAiY291bnQiLAogICAgICAgICJtYXRjaCI6IHsKICAgICAgICAgICJjb3VudCI6IDY0CiAgICAgICAgfQogICAgICB9LAogICAgICB7CiAgICAgICAgInByZWZpeCI6ICJTT1VQX1NUUkVBTUVEX0NFICIsCiAgICAgICAgImV2ZW50IjogInN0cmVhbWluZ19sb3NzX3JldHVybiIsCiAgICAgICAgInBvc2l0aXZlX2ZpZWxkIjogImNvdW50IiwKICAgICAgICAibWF0Y2giOiB7CiAgICAgICAgICAiY291bnQiOiA2NAogICAgICAgIH0KICAgICAgfSwKICAgICAgewogICAgICAgICJwcmVmaXgiOiAiU09VUF9TVFJFQU1FRF9DRSAiLAogICAgICAgICJldmVudCI6ICJjb25zdW1lcl9jb21wbGV0ZSIsCiAgICAgICAgInBvc2l0aXZlX2ZpZWxkIjogImNvdW50IiwKICAgICAgICAibWF0Y2giOiB7CiAgICAgICAgICAiY291bnQiOiA2NAogICAgICAgIH0KICAgICAgfSwKICAgICAgewogICAgICAgICJwcmVmaXgiOiAiU09VUF9DT01QQUNUX01MUCAiLAogICAgICAgICJldmVudCI6ICJvd25lZF9tbHBfZm9yd2FyZCIsCiAgICAgICAgInBvc2l0aXZlX2ZpZWxkIjogImNvdW50IiwKICAgICAgICAibWF0Y2giOiB7CiAgICAgICAgICAiY291bnQiOiAxNzkyLAogICAgICAgICAgIm5hdGl2ZV9mYWxsYmFjayI6IDAsCiAgICAgICAgICAic3RhcnRlcl9vcmlnaW5hbF9jcmVkaXQiOiAwCiAgICAgICAgfQogICAgICB9LAogICAgICB7CiAgICAgICAgInByZWZpeCI6ICJTT1VQX0ZST1pFTl9TVE9SQUdFICIsCiAgICAgICAgImV2ZW50IjogIm93bmVkX2VtYmVkZGluZ19mb3J3YXJkIiwKICAgICAgICAicG9zaXRpdmVfZmllbGQiOiAiY291bnQiLAogICAgICAgICJtYXRjaCI6IHsKICAgICAgICAgICJjb3VudCI6IDY0LAogICAgICAgICAgImluc3RhbGxlZCI6IDEsCiAgICAgICAgICAiaGVhZF9mcDE2X2NhbGxzIjogMCwKICAgICAgICAgICJoZWFkX25hdGl2ZV9vcGVyYW5kX2NhbGxzIjogMCwKICAgICAgICAgICJvd25lcnNfaW50YWN0IjogdHJ1ZSwKICAgICAgICAgICJzdG9yYWdlX29yaWdpbmFsX2NyZWRpdCI6IDAKICAgICAgICB9CiAgICAgIH0sCiAgICAgIHsKICAgICAgICAicHJlZml4IjogIlNPVVBfQUNUSVZBVElPTl9PRkZMT0FEICIsCiAgICAgICAgImV2ZW50IjogIm93bmVkX2FjdGl2YXRpb25fb2ZmbG9hZCIsCiAgICAgICAgInBvc2l0aXZlX2ZpZWxkIjogInBhY2tlZCIsCiAgICAgICAgIm1hdGNoIjogewogICAgICAgICAgImNvdW50IjogNjQsCiAgICAgICAgICAib3duZXJzX2ludGFjdCI6IHRydWUsCiAgICAgICAgICAib3JpZ2luYWxfa2VybmVsX2NyZWRpdCI6IDAKICAgICAgICB9CiAgICAgIH0KICAgIF0sCiAgICAicmVxdWlyZXNfb3JpZ2luYWxfa2VybmVscyI6IHRydWUsCiAgICAiemlwX3NoYTI1NiI6ICIxODBmNWI2Y2Y3NDk3NDRiM2FhZmY5NWY3OGY5ZTFjNTc4NjcwZDgxMzNjZjliMjVmZTRhNzkxM2FkNTJkZDZhIiwKICAgICJ2YWxpZGF0aW9uX3BvbGljeSI6ICJFWFBMSUNJVF9GSU5JVEVfTE9SQV9HUkFESUVOVF9VUERBVEVfV0FSTklOR1NfT05MWSIsCiAgICAic3RhcnRlcl9pbmNsdWRlZCI6IHRydWUsCiAgICAic3RhcnRlcl9vcmlnaW5hbF9jcmVkaXQiOiAwLAogICAgImNvbXBhY3RfbWxwX21lY2hhbmlzbV9hbHJlYWR5X2luX3VzZXJfcGFja2FnZSI6IHRydWUsCiAgICAiZW1iZWRkaW5nX3N0b3JhZ2VfY2hhbmdlZCI6IHRydWUsCiAgICAiZXhjbHVkZWRfa2VybmVscyI6IFsKICAgICAgImV4YW1wbGVfc3dpZ2x1X2Z3ZF9rZXJuZWwiLAogICAgICAiZXhhbXBsZV9zd2lnbHVfYndkX2tlcm5lbCIKICAgIF0sCiAgICAiZW1iZWRkaW5nX29yaWdpbmFsX2NyZWRpdCI6IDAsCiAgICAibmF0aXZlX2VtYmVkZGluZ19sb29rdXAiOiB0cnVlLAogICAgInN0b3JhZ2VfbWVjaGFuaXNtX3ByZWNlZGVudCI6IHRydWUsCiAgICAiaGVhZF9zdHJhdGVneSI6ICJzdHJlYW1lZCIsCiAgICAibnVtZXJpY2FsX2VxdWl2YWxlbmNlIjogIlJFUVVJUkVTX1RBUkdFVF9NRUFTVVJFTUVOVCIKICB9Cn0K', 'streamed_moment_eager_offload_v1.zip': '...', 'streamed_moment_eager_offload_v1_manifest.json': 'ewogICJzb2x1dGlvbi9fX2luaXRfXy5weSI6ICI4MTE2ZWE1OWM0NzJkYjQxZTA5MzBiMjNmMTI5ZjU5NmUxNmExZjA2OTg3MjM0YzBmZTI1YWUzZmM3MTliY2YwIiwKICAic29sdXRpb24vYWN0aXZhdGlvbl9vZmZsb2FkLnB5IjogIjFjNmE5ZDQ5NzQ2YWU0NTFkZWQzZGZkZWI4YWIwYzA4YTczNGIxZjZhYjA2ZTE0NzYxNjkxZTA3NTAxNmVjYmMiLAogICJzb2x1dGlvbi9hY3RpdmF0aW9uX3Byb2JlLnB5IjogIjAzNDRiMDRlNDgzZjI1MjJkZTk1MDc3ZDVkYmI3YmQ2ZTA2NmE0ZTE1YjBhODdlMDJmZjJiNjFhYTUxNTczYTAiLAogICJzb2x1dGlvbi9jZV90aWxlLnB5IjogIjJiYWVhMWY0YjdjNTY3Mzc5ZTM2NzRiNzYyMDRkMzllOWZiMTI4ZWI2Y2ZjZGE5YThkOGVkZWJiZmQyMzBlN2IiLAogICJzb2x1dGlvbi9jaGFpbi5weSI6ICI5OWYwYjE4Y2M5OGRhMDk3MWZiYmRlZGUwYjc5NmI5OGJmZTU0NzJhYTkzZTYxYzlkZTI5YjA5ZWI1MjlkYzJlIiwKICAic29sdXRpb24vY29tbW9uL19faW5pdF9fLnB5IjogImUzYjBjNDQyOThmYzFjMTQ5YWZiZjRjODk5NmZiOTI0MjdhZTQxZTQ2NDliOTM0Y2E0OTU5OTFiNzg1MmI4NTUiLAogICJzb2x1dGlvbi9jb21tb24vYnJpZGdlX3N0YXRlLnB5IjogImUzYmE2NDQ0NmE4YmIzZWI4Y2Q2ZWMzMWI0ZTVkMDAxOGE0N2M4ZjNiOTBkMDBkYTk1NWRjMmU0NDRlYjE4ZTkiLAogICJzb2x1dGlvbi9jb21tb24vY29uc3VtZXIucHkiOiAiZjAxOGE5YjY4MDhkNzYzODFhMjA1NTgwZThhMmZmOWQyYzBlYjcwNmZhMzRjODQzNGZlNDFiMGFkOTlkYTJiYyIsCiAgInNvbHV0aW9uL2NvbXBhY3RfbWxwX2luc3RhbGwucHkiOiAiNGM3OTYwNjQ5ZGFhZjY5MGMyM2JlYjc0N2Y4NTE3M2NjZDViMjQ1OGYwNjljMWIzNjYwZjBjMTU1YzU0NzNkNCIsCiAgInNvbHV0aW9uL2NvbnRyYWN0cy5weSI6ICIzNmJiNTRjNmNhYzQwYjdmMzg5NmIyY2UzYTUyNjY1Zjk3Mjk0MmY1ZTU4NTZlMjM3OTQ2YTFmODAwOTFmYjRiIiwKICAic29sdXRpb24vZXhwbG9yYXRvcnlfcHJlZmxpZ2h0LnB5IjogIjllNDZiNWRkMjY5N2U3ZmI0MmMwMDUzYjRjMDc2MmQ0NjU2MTAyM2NmYjM5OGZiNzlkMzZiMzAyNjc0MTJlMDYiLAogICJzb2x1dGlvbi9mcm96ZW5fc3RvcmFnZS5weSI6ICJjMmQ1NWQzM2U3ZjEzNDQ4NGQxYjRlMjUxOGE2NDkxNTJkNTJhYWNmYmU4MjQwYTdkODAxYzhmNGI1MmFjNjYzIiwKICAic29sdXRpb24vbWxwX2hhbGYvX19pbml0X18ucHkiOiAiYzhlMjVjODNkYTc3NjQ5YzdlNzZkN2UyNTU3ZGNhOGEzYjllOTc1YTgxZmE3Mzg3MTMyMTFhOWE4ZjhkNmZmMCIsCiAgInNvbHV0aW9uL21scF9oYWxmL25hdGl2ZV9vd25lci5weSI6ICI3YjY5ZjJhMzU0ZGVmMjVjYjUzZDRjYTc4ZjNjMTkwODI0OTc4MzhjNWY2ZWUxMzNkOTQ2OTRlNjExM2Q1NDBjIiwKICAic29sdXRpb24vbWxwX2hhbGYvb3duZWRfaGFsZi5weSI6ICI4YTcxNDIwNWM0ZjFkZThmZjc5OTdmOTQzMGU0N2ZiNDE3YjU0ZTY2ODc5YjY0ZmE4MDE0NDg5NmM1MmE4MDdiIiwKICAic29sdXRpb24vbWxwX2hhbGYvc3RhcnRlcl9oYWxmLnB5IjogIjM3NTlmMzc2MWIzYjNkMmE4ZTAxOGI4MGE4NTcxYzljYzE0NTZiOGFkYzFlZDQwZTQ3ZTFjOTE5MTMyZTdkZGEiLAogICJzb2x1dGlvbi9tbHBfaGFsZi9zdGFydGVyX3N3aWdsdS5weSI6ICJlNjY1OTEwZmFlY2E2MDkxYzk1OGVkN2ExOTE2MDQ0ODRhNDQyMTZmNTFkNmZhMDY5ZjVmNDgyMjBlNzQ3MmU2IiwKICAic29sdXRpb24vbW9tZW50X3RpbGUucHkiOiAiZDVhMmI5NWEwMDQ3Mjc2MmYwY2ViYjE1OTA2ZDcyMWZlMjY0NWExNTdjYjc3MmY0NjhkOGIzODFlNDY0YmY3ZCIsCiAgInNvbHV0aW9uL29mZmxvYWRfcHJlZmxpZ2h0LnB5IjogIjU4MmY2YTlhMTRkOThiZTNhNjQxYmIxYmNjNzcxODI2NWNiZjBmYTdkZWU3YWFkMzc4OThhMGZjMWFhOWMwYjkiLAogICJzb2x1dGlvbi9vcmlnaW5hbC9fX2luaXRfXy5weSI6ICJlM2IwYzQ0Mjk4ZmMxYzE0OWFmYmY0Yzg5OTZmYjkyNDI3YWU0MWU0NjQ5YjkzNGNhNDk1OTkxYjc4NTJiODU1IiwKICAic29sdXRpb24vb3JpZ2luYWwvbWF0ZXJpYWxpemVkX2NlLnB5IjogIjY1MTljOGYxZGRjZTdkYjI3MGZiZTQyNWE4ZTg3NzFhZWNkYTNmMjk0ZWNjODM4MzlkNDc1OTUzMWZkMWRjZTEiLAogICJzb2x1dGlvbi9vcmlnaW5hbC9tZXRyaWNfc3VtbWFyeS5weSI6ICI4YzVlZGVhMDBjN2YwYjQyNTAxYzcxZTM0YTk4NmRlMDVjMjhjM2M1NmVkZjRkNjE4MWQyZGRhZjI0MTQyY2Y0IiwKICAic29sdXRpb24vb3JpZ2luYWwvbWl4ZWRfcm93cy5weSI6ICIwOTExNTdjZGI5MDRiMGEyZmVlMDlkNzIwODYyYjM3MGQ4ZjcyZjlkODI0MDIzZDM3ZWRiZGY5YzVlMTcyNmIyIiwKICAic29sdXRpb24vb3duZXJzaGlwLnB5IjogIjc4NGM1NmRkNWMwYzhmNmRjYTEzMTFiMzhhNzFmMDdkNzgwOTgyNGFiOGNlMzM0NTY1NzkwMDViMWU3NzZmMDUiLAogICJzb2x1dGlvbi9wcmVwYXJlX3JldGFpbmVkLnB5IjogIjE5ZTFmMmIxYjU4ODVkYjcxZDE1MmU2ZWNjMGQ0YTlkNWQ0YmI3NjdiNjZlYTRjZmUzZWExZTQxYmEyZWFlNTEiLAogICJzb2x1dGlvbi9zb3VyY2Vfb3duZXJzLnB5IjogImM3MDFiZmQ3MDYzZTI4OWQ5MWM4ZTViODU2YjYyODI3ZjZhOWY0ZjI1NzlhYTIzYmJlZDRhOTExYTVkMzhlNWIiLAogICJzb2x1dGlvbi9zdHJlYW1lZF9tb21lbnRfaGVhZC5weSI6ICJmMmNlYWIwMTk4OTEzYWUwMGNhNzRkNzQ2YWE0YmQ3OGFjZDI0OGNjOTZjZGJlNGQzZWYxYWJiYWY4MTZjN2FiIiwKICAic29sdXRpb24vc3RyZWFtaW5nX2JvdW5kYXJ5LnB5IjogIjc1ZmU4YmY5NTgyMTNhMzc4MmVhZmEwMjNlNGEzZDFjZDQ3Y2RjYTc0ODM1NTYyYmQ4MWFkNzgyN2E5NjQ4YWEiLAogICJzb2x1dGlvbi9zdXBwb3J0L19faW5pdF9fLnB5IjogImUzYjBjNDQyOThmYzFjMTQ5YWZiZjRjODk5NmZiOTI0MjdhZTQxZTQ2NDliOTM0Y2E0OTU5OTFiNzg1MmI4NTUiLAogICJzb2x1dGlvbi9zdXBwb3J0L25mNC5weSI6ICI5ZTA3YTczZDQ1ZjdiNzExOWZjMWY1ZDFkMzQyMjM1MWU1ZDM0ZTJmYmQ5NjEyZjdiOTBmNTZiYzE3YWQ2OTdhIiwKICAic29sdXRpb24vc3VwcG9ydC9ybXNub3JtLnB5IjogImFhNTVhMjFmZDkyMWQyZjU0Nzg0NDRiMDNhMzI2OTRhMzI5YmIxMGJiYmU0ZjNiZjQ2ZTRlNzhmMjgwNGE5M2MiLAogICJzb2x1dGlvbi9zdXBwb3J0X2luc3RhbGwucHkiOiAiY2MyYjQwMDU5N2JiMzUyNDZjMzUzODkzMTg1YzM3MDg3Mzk0YmI2YzBlNWU5YTNhNjc3ZjIxZGMyYmYzODk4NCIsCiAgInNvbHV0aW9uL3RlbGVtZXRyeS5weSI6ICI4NmZiZjczZGYwN2M5MTc5NDc0YTk0ZjJjNTM0OWYwNGQ4YTM2ZGYxNmVkM2UwODRlYzk0MzNjYjMwNTU2NjJjIiwKICAic29sdXRpb24vdG9rZW5fY2h1bmtfaGVhZC5weSI6ICJiY2M2NDE3ZTRkMzc2OTIwNjY4YmNiMzQxZjU0OTFhZmUzMjg0N2Q3NjhhN2QxMGRmNzU1MDNiMWM5ZTI0NGFjIiwKICAic29sdXRpb24vdmVyaWZ5X3RpbGUucHkiOiAiZjRmNjI1MTk4N2I2MTk5MzY1Yjg1M2Q0ODk3OWExMzZlYzJkYzI2MjZiZWQ2OGYzNzRhZTI5NDBlZTYwNDdhMSIsCiAgInNvbHV0aW9uL3dlYl9wcmVmbGlnaHQucHkiOiAiYjFiN2IzOTQyODAxY2VlMjE2NGU3ZjNmNWZhNzhkNzk3ZmFlZDI4ZDE1ZjJkY2NjMDk2M2I0MTg2MzJhMzE5NSIKfQo='}
PAYLOAD_SHA256 = {'README.md': '0314d2bdcddbc4d9eebd86772011685a133fb8af873ed5129a91dbddf1eb812a', 'runtime_api_reference.json': 'd79e555702154a0516efa87ca22ab00dad39c515eefb12fc7332a0da40108881', 'owned_process_tree.py': '693dde708da5c06053af239463952115ef5e8865f50bcbc51a2038237ca57d69', 'runtime_probe.py': 'efe1b7cf3f3cf1e37bbc169a8468859c5127ec3281a37e39068703b2eb3c1480', 'prepare_asset_manifest.py': 'f2a0f36b5cd367d506c831c69f1a39d49c38c3c815758be5d0fd7a2ec033f46d', 'process_guard.py': '82465943a3a9af3831a5399a3a55d09ac290ba7365c4e80d6bc5e79fab62a6ff', 'evidence_reader.py': '95a3e3b9956bae7b998cd528c1e8e3023eda3a0a4f006ef29ca224eb8337b03c', 'web_runner.py': '4b879032a2f8e55df581cae6420282e9534105726d969a9c21696dab665ac549', 'candidate_profile.json': '6a2a8a81c2f33b59adc9d4404d219ee31ed655fea14f1adede69467b0671eca0', 'streamed_moment_eager_offload_v1.zip': '180f5b6cf749744b3aaff95f78f9e1c578670d8133cf9b25fe4a7913ad52dd6a', 'streamed_moment_eager_offload_v1_manifest.json': '306534d285c65eb7ced8abe92f09b61558a10b8f0c459b2e39b7d8f4b2263a0f'}
STAGED = Path(tempfile.mkdtemp(prefix='soup-web-'+MODE+'-', dir='/kaggle/working'))
for name, encoded in PAYLOAD.items():
    raw = base64.b64decode(encoded, validate=True)
    assert hashlib.sha256(raw).hexdigest() == PAYLOAD_SHA256[name], name
    (STAGED/name).write_bytes(raw)
print('Prepared mode:', MODE)
print('Official source will be verified by bytes, not .git.')
print('Requested native training cap: 80 seconds; no external-cap bypass.')


## Execute the authorized prepared-job screen
The notebook process imports no Torch and keeps no CUDA context. Short-lived probe/preflight children exit before idle measurement. Full task/LoRA/data/precision remain unchanged. Small evidence files export incrementally below SOUP_RESULTS; adapters and compiler caches stay local. An external whole-job kill can leave status RUNNING: that is censored, never success.


In [ ]:
import subprocess, sys, signal
# Executing/submitting this notebook is the GPU action. Preparation did not run it.
# Never force-kill this supervisor: its guardian owns descendant cleanup and lock.
process = subprocess.Popen([sys.executable, '-B', str(STAGED/'web_runner.py'), '--mode', MODE], start_new_session=True)
try:
    return_code = process.wait()
except BaseException:
    if process.poll() is None:
        process.send_signal(signal.SIGTERM)
    while process.poll() is None:
        try: process.wait()
        except KeyboardInterrupt: pass
    raise
if return_code:
    raise RuntimeError('Screen blocked, failed, or censored. Inspect exported status/logs; do not treat it as a score or automatically retry.')
